In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association

url = "D:/04_IronHack_DA/IRONHACK_Labs/amz_uk_price_prediction_dataset.csv"
df = pd.read_csv(url)
df.head(3)

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers


In [ ]:
#Part 1: Analyzing Best-Seller Trends Across Product Categories
#Objective: Understand the relationship between product categories and their best-seller status.

#Crosstab Analysis:
crosstab_result = pd.crosstab(df['category'], df['isBestSeller'])
crosstab_result
bestsellers_proportion=df.groupby('category')["isBestSeller"].mean().sort_values(ascending=False)
bestsellers_proportion
# Categoreies like Grocery, Smart Home Security & Lighting and Health & Personal Care are more prevalent best-seller categories

#Statistical Tests:
chi2_statistic, chi2_p_value, _, _ = chi2_contingency(crosstab_result)

print(f"Chi-square statistic: {chi2_statistic:.2f}")
print(f"p-value: {chi2_p_value:.2f}")

association(crosstab_result, method="cramer")
# The chi squre test shows that the p-value is 0 which means that bestseller is not independent of the product category.
# caremer's test suggest that there is some relation between them but its not strong.

#Visualizations:
crosstab_result.head(30).plot(kind="bar", stacked=True)

In [ ]:
crosstab_result = pd.crosstab(df['category'], df['isBestSeller'])
crosstab_result.head()

In [ ]:
#Part 2: Exploring Product Prices and Ratings Across Categories and Brands
#Objective: Investigate how different product categories influence product prices.

#Preliminary Step: Remove outliers in product prices.
def remove_outliers(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data >= lower_bound) & 
        (data <= upper_bound)]
    
    return outliers

no_outlier = remove_outliers(df["price"])
df.loc[no_outlier.index]

#Violin Plots:
top_20_categories = df["category"].value_counts().head(20).index
df_top_20 = df[df["category"].isin(top_20_categories)]

price_no_outliers_top20 = remove_outliers(df_top_20["price"])

df_top_20_no_outliers = df_top_20.loc[price_no_outliers_top20.index]

sns.violinplot(data=df_top_20_no_outliers, x='price', y='category', palette="coolwarm")
# Men'S category have the highest median price. Although Iam not sure of it because men and fragrances look almost the same.

#Bar Charts:
mean_price_top_10 = (df.groupby("category")["price"].mean().head(10))

mean_price_top_10

mean_price_top_10.plot.bar()
# 3D printers have the highest average price


#Box Plots:
top_10_categories = df["category"].value_counts().head(10).index
df_top_10 = df[df["category"].isin(top_10_categories)]
df_top_10["category"]("stars").median

sns.boxplot(data=df_top_10_no_outliers, x='stars', y='category', palette="coolwarm")

# skincare, bath & body, beauty, hair care received the highest median ratings